In [1]:
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ================= 配置 =================
DATASET_ROOT = "./dataset_STFT_Tensor"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

# 真实数据路径
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "16QAM",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "64QAM",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/L20250425_103838_F6000000000_S400000_B100000_M0_IQ_ID0_nozaoshen_100ksps_QPSK.dat": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/L20250425_104306_F6000000000_S400000_B100000_M0_IQ_ID0_nojiazao_100ksps_OQPSK.dat": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

# STFT 参数：保证输出接近 64x64
SAMPLES_LEN = 1024 # 切片长度
N_FFT = 64         # 频域分辨率 -> 33 bins (Resize to 64)
HOP_LEN = 16       # 时域分辨率 -> 1024/16 = 64 frames
IMG_SIZE = 64

TOTAL_SIM_SAMPLES_PER_CLASS = 1000
TRAIN_VAL_RATIO = 0.8
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# --- 引入之前的仿真信号生成函数 ---
# (为了节省篇幅，请务必把之前 data_generator_rad_final.py 里的 
#  generate_rrc_filter, apply_channel_effects, generate_sim_signal 
#  这三个函数完整复制到这里！)
# -----------------------------------------------------------
# [在此处粘贴 generate_sim_signal 等函数]
# 确保 generate_sim_signal 内部保留了 step = np.random.choice([1, 2, 4]) 的降采样逻辑
# -----------------------------------------------------------
# <占位符: 请复制粘贴>
def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def apply_channel_effects(sig, snr_db):
    N = len(sig)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    phase_jitter = torch.randn(N) * 0.005 
    sig = sig * torch.exp(1j * phase_jitter)
    g_imbalance = np.random.uniform(0.98, 1.02)
    p_imbalance = np.random.uniform(-1, 1) * np.pi/180
    real = sig.real * g_imbalance
    imag = sig.imag * np.cos(p_imbalance) - sig.real * np.sin(p_imbalance)
    sig = torch.complex(real, imag)
    sig_power = torch.mean(torch.abs(sig)**2)
    real_snr = snr_db + 5 
    noise_power = sig_power / (10**(real_snr/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

def get_32qam_constellation():
    points = []
    for x in [-5, -3, -1, 1, 3, 5]:
        for y in [-5, -3, -1, 1, 3, 5]:
            if not (abs(x) == 5 and abs(y) == 5): points.append(x + 1j*y)
    return torch.tensor(points, dtype=torch.complex64)

def get_apsk_symbols(M, ratios, num_symbols):
    if M == 16:
        R1, R2 = 1.0, ratios[0]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_outer = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        constellation = torch.cat([pts_inner, pts_outer])
    elif M == 32:
        R1, R2, R3 = 1.0, ratios[0], ratios[1]
        pts_inner = R1 * torch.exp(1j * (torch.arange(4)*2*np.pi/4 + np.pi/4))
        pts_mid = R2 * torch.exp(1j * (torch.arange(12)*2*np.pi/12))
        pts_outer = R3 * torch.exp(1j * (torch.arange(16)*2*np.pi/16))
        constellation = torch.cat([pts_inner, pts_mid, pts_outer])
    indices = torch.randint(0, M, (num_symbols,))
    return constellation[indices]

def generate_sim_signal(mod_type, N, snr_db):
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs) 
    num_symbols = int(N / sps) + 50
    syms = None
    if mod_type == "BPSK": syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK": 
        idx = torch.randint(0, 4, (num_symbols,))
        syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
    elif mod_type == "8PSK":
        idx = torch.randint(0, 8, (num_symbols,))
        syms = torch.exp(1j * idx * 2*np.pi/8)
    elif mod_type == "OQPSK":
        idx = torch.randint(0, 4, (num_symbols,))
        syms = torch.exp(1j * (np.pi/4 + idx * np.pi/2))
    elif mod_type == "PI4DQPSK":
        bits = torch.randint(0, 4, (num_symbols,))
        phase_shifts = torch.tensor([np.pi/4, 3*np.pi/4, -np.pi/4, -3*np.pi/4])[bits]
        phases = torch.cumsum(phase_shifts, dim=0)
        syms = torch.exp(1j * phases)
    elif mod_type == "2ASK":
        bits = torch.randint(0, 2, (num_symbols,)).float()
        syms = bits + 0j
    elif mod_type == "16QAM":
        idx = torch.randint(0, 16, (num_symbols,))
        i = (idx // 4) * 2 - 3; q = (idx % 4) * 2 - 3; syms = i + 1j*q
    elif mod_type == "32QAM":
        constellation = get_32qam_constellation()
        idx = torch.randint(0, 32, (num_symbols,))
        syms = constellation[idx]
    elif mod_type == "64QAM":
        idx = torch.randint(0, 64, (num_symbols,))
        i = (idx // 8) * 2 - 7; q = (idx % 8) * 2 - 7; syms = i + 1j*q
    elif mod_type == "16APSK": syms = get_apsk_symbols(16, [2.7], num_symbols)
    elif mod_type == "32APSK": syms = get_apsk_symbols(32, [2.6, 4.5], num_symbols)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000) 
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        sig = apply_channel_effects(sig, snr_db)
        step = np.random.choice([1, 2, 4])
        sig_down = sig[::step]
        if len(sig_down) < N:
            repeats = N // len(sig_down) + 1
            sig_down = sig_down.repeat(repeats)[:N]
        return sig_down

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    if mod_type == "OQPSK":
        upsampled_i = torch.zeros(num_symbols * sps); upsampled_q = torch.zeros(num_symbols * sps)
        upsampled_i[::sps] = syms.real
        q_temp = torch.zeros(num_symbols * sps); q_temp[::sps] = syms.imag
        upsampled_q = torch.roll(q_temp, shifts=sps//2); upsampled_q[:sps//2] = 0
        upsampled = upsampled_i + 1j * upsampled_q
    else: upsampled[::sps] = syms

    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    
    if len(sig) > N: sig = sig[:N]
    else: sig = F.pad(sig, (0, N - len(sig)))
    sig = apply_channel_effects(sig, snr_db)
    
    step = np.random.choice([1, 2, 4])
    sig_down = sig[::step]
    if len(sig_down) < N:
        repeats = N // len(sig_down) + 1
        sig_down = sig_down.repeat(repeats)[:N] 
    else: sig_down = sig_down[:N]
    return sig_down

# ================= 2. 核心生成逻辑: STFT Tensor =================

def generate_stft_tensor(iq_complex, target_size=(64, 64)):
    """
    输入: (1024,) 复数 Tensor
    输出: (3, 64, 64) Float Tensor
         Ch1: Log Magnitude (Spectrogram)
         Ch2: Phase Angle
         Ch3: Differential Phase
    """
    if iq_complex.dim() > 1: iq_complex = iq_complex.squeeze()
    
    # 1. 计算 STFT
    # return_complex=True (PyTorch > 1.7)
    Z = torch.stft(iq_complex, n_fft=N_FFT, hop_length=HOP_LEN, return_complex=True)
    # Z shape: (Freq_bins, Time_frames) approx (33, 65)
    
    # 2. 提取特征
    # [Channel 1] Log Magnitude
    mag = torch.abs(Z)
    log_mag = torch.log1p(mag)
    
    # [Channel 2] Phase
    phase = torch.angle(Z) # (-pi, pi)
    
    # [Channel 3] Differential Phase (沿时间轴差分)
    # 捕捉相位跳变 (QPSK killer feature)
    diff_phase = torch.diff(phase, dim=1, prepend=phase[:, :1])
    # 将差分值折叠回 (-pi, pi)，消除卷绕影响
    diff_phase = (diff_phase + np.pi) % (2 * np.pi) - np.pi
    
    # 3. 堆叠
    # shape: (3, Freq, Time)
    tensor_3ch = torch.stack([log_mag, phase, diff_phase], dim=0)
    
    # 4. 调整尺寸到 64x64
    # unsqueeze(0) for batch dim -> interpolate -> squeeze
    tensor_resized = F.interpolate(tensor_3ch.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)
    
    # 5. 归一化 (Instance Normalization)
    # 简单的 Min-Max 或 Standardize，这里用简单的除以最大值
    for c in range(3):
        mx = torch.max(torch.abs(tensor_resized[c]))
        if mx > 1e-6:
            tensor_resized[c] = tensor_resized[c] / mx
            
    return tensor_resized

# ================= 主流程 =================
if __name__ == "__main__":
    if os.path.exists(DATASET_ROOT): shutil.rmtree(DATASET_ROOT)
    print("===== Generating STFT (Amp/Phase/Diff) Tensor Data =====")
    
    # 1. 仿真
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        os.makedirs(os.path.join(SIM_VAL_DIR, cls_name), exist_ok=True)
        print(f"Generating {cls_name}...")
        
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            target_dir = SIM_TRAIN_DIR if np.random.rand() < TRAIN_VAL_RATIO else SIM_VAL_DIR
            target_dir = os.path.join(target_dir, cls_name)
            snr = np.random.uniform(-5, 25)
            
            # 生成 (长度 1024)
            # 注意：generate_sim_signal 内部会做随机 step=1/2/4
            sig = generate_sim_signal(cls_name, SAMPLES_LEN, snr)
            if isinstance(sig, np.ndarray): sig = torch.from_numpy(sig)
            
            # STFT 转换
            tensor = generate_stft_tensor(sig.to(DEVICE))
            torch.save(tensor.cpu(), os.path.join(target_dir, f"{cls_name}_{i:04d}_{snr:.1f}.pt"))

    # 2. 真实数据
    print("\nProcessing Real Data...")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): continue
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        try:
            if file_path.endswith('.dat'):
                raw = np.fromfile(file_path, dtype='>i2')
                if len(raw) > 1024: raw = raw[1024:]
            else:
                raw = np.fromfile(file_path, dtype=np.int16)
            
            if raw.size % 2 != 0: raw = raw[:-1]
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            iq_complex = (I + 1j * Q) / 32767.0
            
            # 切片 (滑动窗口)
            num_slices = len(iq_complex) // SAMPLES_LEN
            saved_cnt = 0
            for i in range(num_slices):
                if saved_cnt >= 500: break
                
                chunk_np = iq_complex[i*SAMPLES_LEN : (i+1)*SAMPLES_LEN]
                
                # [关键] 真实数据也做随机降采样增强？不，真实数据做多尺度测试更好
                # 但为了生成数据集方便，这里我们生成 3 个副本 (1x, 2x, 4x)
                # 或者简单点，我们生成一个强制 4x 的版本（针对坏 QPSK）和一个 1x 的版本
                # 鉴于之前结论，坏 QPSK 必须 4x 才能看清
                # 我们这里采用：只生成 4x 版本 (step=4) 作为测试集主力
                # 因为 1x 版本对于高 SPS 信号来说特征太糊了
                
                chunk_np = chunk_np[::4] # 强制降采样，显露特征
                
                # 补齐
                if len(chunk_np) < SAMPLES_LEN:
                     repeats = SAMPLES_LEN // len(chunk_np) + 1
                     chunk_np = np.tile(chunk_np, (repeats,))[:SAMPLES_LEN]
                
                chunk_tensor = torch.from_numpy(chunk_np).to(DEVICE)
                tensor = generate_stft_tensor(chunk_tensor)
                torch.save(tensor.cpu(), os.path.join(save_dir, f"real_{i:04d}.pt"))
                saved_cnt += 1
                
        except Exception as e: print(f"Error {label}: {e}")
            
    print(f"\nDone! Dataset saved at {DATASET_ROOT}")

d:\anaconda3\envs\pytorch1_12_1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===== Generating STFT (Amp/Phase/Diff) Tensor Data =====
Generating BPSK...


100%|██████████| 1000/1000 [00:02<00:00, 401.87it/s]


Generating QPSK...


100%|██████████| 1000/1000 [00:01<00:00, 734.70it/s]


Generating 8PSK...


... (16 outputs left out)

100%|██████████| 1000/1000 [00:01<00:00, 563.35it/s]


Generating FSK2...


100%|██████████| 1000/1000 [00:01<00:00, 665.07it/s]



Processing Real Data...

Done! Dataset saved at ./dataset_STFT_Tensor
